In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd 
import os

df = pd.read_csv('d_norm_train_t.csv')
col_names = df.columns.tolist()
df.head()


In [ ]:
X = df.drop('class', axis = 1)
y = df['class']

In [ ]:
X.shape

In [ ]:
X = X.values 
y = y.values
def split_data_by_class(X, y):
    class_data = {}
    for class_label in np.unique(y):
        indices = np.where(y == class_label)[0]
        class_data[class_label] = X[indices]
    return class_data

class_data = split_data_by_class(X, y)


In [ ]:
class_data[4].shape


In [ ]:
def make_generator_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(256, input_dim=100),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.LeakyReLU(),
        tf.keras.layers.Dense(512),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.LeakyReLU(),
        tf.keras.layers.Dense(1024),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.LeakyReLU(),
        tf.keras.layers.Dense(23, activation='sigmoid')  
    ])
    return model


In [ ]:
def make_discriminator_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(512, input_shape=(23,)),
        tf.keras.layers.LeakyReLU(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(256, ),
        tf.keras.layers.LeakyReLU(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(128,),
        tf.keras.layers.LeakyReLU(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    return model


In [ ]:
generator = make_generator_model()
discriminator = make_discriminator_model()

generator_optimizer = tf.keras.optimizers.legacy.Adam(1e-4)
discriminator_optimizer = tf.keras.optimizers.legacy.Adam(1e-4)


In [ ]:
cross_entropy = tf.keras.losses.BinaryCrossentropy()

def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    total_loss = real_loss + fake_loss
    return total_loss

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)


In [ ]:
from IPython.display import display, clear_output
from tqdm import tqdm

noise_dim = 100
num_examples_to_generate = 16  
batch_size = 128
seed = tf.random.normal([num_examples_to_generate, noise_dim])

@tf.function
def train_step(data):
    noise = tf.random.normal([batch_size, noise_dim])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_data = generator(noise, training=True)

        real_output = discriminator(data, training=True)
        fake_output = discriminator(generated_data, training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))


checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(generator=generator,
                                 discriminator=discriminator,
                                 gen_optimizer= generator_optimizer,
                                 disc_optimizer=discriminator_optimizer)



def train(dataset, epochs):
    total_batches = len(dataset) * epochs
    overall_progress = tqdm(total=total_batches, desc='Overall Progress', unit='batch')
    
    for epoch in range(epochs):
        for image_batch in dataset:
            train_step(image_batch)
            overall_progress.update(1)
        clear_output(wait=True)
        # Update the overall progress
        #progress_bar = tqdm(dataset, desc=f'Epoch {epoch + 1}/{epochs}', unit='batch')
        # for image_batch in progress_bar:
        #     train_step(image_batch)
        generate_and_save_data(generator, epoch + 1, seed)

        # Save the model every 15 epochs
        # if (epoch + 1) % 15 == 0:
            # checkpoint.save(file_prefix = checkpoint_prefix)
    overall_progress.close()
    
    # Generate after the final epoch
    generate_and_save_data(generator, epochs, seed)

In [ ]:
def prepare_dataset_for_class_2(class_features, batch_size=512, buffer_size=10000):
    dataset = tf.data.Dataset.from_tensor_slices(class_features)
    dataset = dataset.shuffle(buffer_size).batch(batch_size)
    return dataset

def prepare_dataset_for_class_3(class_features, batch_size=32, buffer_size=10000):
    dataset = tf.data.Dataset.from_tensor_slices(class_features)
    dataset = dataset.shuffle(buffer_size).batch(batch_size)
    return dataset

def prepare_dataset_for_class_4(class_features, batch_size=2, buffer_size=10000):
    dataset = tf.data.Dataset.from_tensor_slices(class_features)
    dataset = dataset.shuffle(buffer_size).batch(batch_size)
    return dataset



In [ ]:
dataset_class_2 = prepare_dataset_for_class_2(class_data[2])
dataset_class_3 = prepare_dataset_for_class_3(class_data[3])
dataset_class_4 = prepare_dataset_for_class_4(class_data[4])


In [ ]:
import numpy as np
import pandas as pd
import os

def generate_and_save_data(model, epoch, test_input, save=True, path='numerical_data'):
    predictions = model(test_input, training=False)

    predictions_df = pd.DataFrame(predictions.numpy())

    if save:
        if not os.path.exists(path):
            os.makedirs(path)
        file_path = os.path.join(path, f'data_at_epoch_{epoch:04d}.csv')
        predictions_df.to_csv(file_path, index=False)

    print(f"Predictions for epoch {epoch} saved to {file_path}.")


In [ ]:
def generate_synthetic_data(model, num_samples):
    noise = tf.random.normal([num_samples, noise_dim])
    generated_data = model(noise, training=False)
    return generated_data.numpy()

num_samples_needed_2 = 45927 - len(class_data[2])  
num_samples_needed_3 = 45927 - len(class_data[3])  
num_samples_needed_4 = 45927 - len(class_data[4])  


In [ ]:
train(dataset_class_2, 50)
synthetic_data_class_2 = generate_synthetic_data(generator, num_samples_needed_2)



In [ ]:
train(dataset_class_3, 50)
synthetic_data_class_3 = generate_synthetic_data(generator, num_samples_needed_3)


In [ ]:

train(dataset_class_4, 50)
synthetic_data_class_4 = generate_synthetic_data(generator, num_samples_needed_4)


In [ ]:
X.shape

In [ ]:
def combine(class_num, syn_class, num_samples):
    x_syn = syn_class
    y_syn = np.array([class_num] * num_samples)
    global X 
    X= np.concatenate([X, x_syn], axis = 0)
    global y 
    y = np.concatenate([y, y_syn], axis = 0)

def fuse(x_df, y_df):
    xy_fuse = np.column_stack((X, y))
    global df_new
    df_new = pd.DataFrame(xy_fuse)
    df_new.columns = col_names
    

In [ ]:
combine(2, synthetic_data_class_2, num_samples_needed_2)
combine(3, synthetic_data_class_3, num_samples_needed_3)
combine(4, synthetic_data_class_4, num_samples_needed_4)

In [ ]:
fuse(X, y)
df_new.shape


In [ ]:

#df_new.to_csv('df_aug.csv', index = False)

In [ ]:
df_new['class'].value_counts()